# One-class autoencoder on the T = 10 benign windows of all 8 scenario folders

The one-class baseline of `task/autoencoder_benign_t10` at T = 10: the same small 1D-conv autoencoder (14 × 10
in, z = 16) with 1 stride-2 stage(s) instead of 2 so that 10 divides (encoder strides 1, 2, 1; decoder ConvTranspose k3 s1 (length kept), k4 s2; 52,861 parameters),
trained on **genuine (benign) windows only** from `src/data/encoder_input/all/T10_benign/`. The train / val /
test split (80 / 10 / 10 by sender vehicle over all 8 folders, the all/T10_benign split) is fixed on disk; train
and val hold benign windows only. Val drives early stopping, the LR schedule and the alarm thresholds. The test
splits are opened only in section 5: the benign test split and the attack windows of the same test vehicles
(`test_gridsybil`, `test_datareplay`, `test_dosrandom`, `test_dosdisruptive`). Every training setting equals the
T = 32 package's.

Two label-free scores per window: the reconstruction error (masked MSE over real rows), and the **Mahalanobis
distance** of z to a Gaussian fitted on the benign train z. Each is also aggregated per **track** (one link = one
receiver hearing one sender pseudonym): max and mean over the track's windows.

This notebook runs from the repo root, imports the package classes and reads the run folder written by
`npm run aeb10:train`. Sections: 1 input, 2 model, 3 training record, 4 reconstruction on val, 5 test, 6 context,
7 analysis. A side experiment, not a thesis result, not the thesis TimesNet model.

Setup: repo-root check, package imports, the run folder (the newest `runs/*/summary.json`).

In [ ]:
import json
from pathlib import Path

assert Path("task/autoencoder_benign_t10/config.py").is_file(), "run this notebook from the repo root"

from task.autoencoder_benign_t10.config import EVAL_SPLITS, TOTAL_PARAMS, AutoencoderConfig
from task.autoencoder_benign_t10.data import WindowDataset
from task.autoencoder_benign_t10.model import ConvAutoencoder, count_params
from task.autoencoder_benign_t10.scoring import Mahalanobis, TrackScores, quantile_thresholds
from task.autoencoder_benign_t10.trainer import Trainer

%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.decomposition import PCA
from sklearn.metrics import average_precision_score, roc_auc_score

settings = AutoencoderConfig()
runs = sorted(Path(settings.runs_dir).glob("*/summary.json"), key=lambda p: p.stat().st_mtime)
assert runs, "no finished run: npm run aeb10:train"
RUN = runs[-1].parent
META = json.loads((Path(settings.data_dir) / "metadata.json").read_text())
FEATURES = META["features"]
ATTACKS = {
    "test_gridsybil": "GridSybil",
    "test_datareplay": "DataReplay",
    "test_dosrandom": "DoSRandom",
    "test_dosdisruptive": "DoSDisruptive",
}
print("run folder:", RUN)
print("data:", settings.data_dir)

## 1. Input

Seven splits on disk. Train / val / test hold the **benign** windows of disjoint sender-vehicle sets (80 / 10 / 10 by
vehicle key `<group>:<vehicle>`, stratified by group and vehicle class, seed 0); the four `test_<attack>` splits hold the
attack windows of the **test** vehicles (attack windows of train / val vehicles were dropped when the input was built).
Counts below come from `metadata.json`; the test splits themselves are not opened before section 5.

In [ ]:
order = ["train", "val", "test", *ATTACKS]
cbs = pd.DataFrame(META["counts_by_split"]).T.loc[order]
cbs = cbs[["windows", "messages", "links", "vehicles", "padding_share", "windows_1_3", "share_1_3"]]
for c in ("windows", "messages", "links", "vehicles", "windows_1_3"):
    cbs[c] = cbs[c].astype(int)
cbs.columns = [
    "windows",
    "messages",
    "links (tracks)",
    "sender vehicles",
    "padding share",
    "1-3-row windows",
    "1-3-row share",
]
cbs

Split diagram: windows per split (log scale), with the sender vehicles behind each split.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.8))
colors = ["tab:blue", "tab:cyan", "tab:green", "tab:orange", "tab:red", "tab:purple", "tab:brown"]
for y, name in enumerate(reversed(order)):
    w = int(cbs.loc[name, "windows"])
    ax.barh(y, w, color=colors[order.index(name)])
    ax.text(
        w * 1.08,
        y,
        f"{w:,} windows, {int(cbs.loc[name, 'sender vehicles']):,} vehicles, "
        f"1-3-row share {cbs.loc[name, '1-3-row share']:.1%}",
        va="center",
        fontsize=8,
    )
ax.set_yticks(range(len(order)), list(reversed(order)))
ax.set_xscale("log")
ax.set_xlim(1e4, 3e7)
ax.axhline(3.5, color="black", lw=0.8)
ax.text(1.1e4, 3.6, "benign only: train fits the model, val selects it", fontsize=8, va="bottom")
ax.text(1.1e4, 3.4, "evaluation only: test vehicles (benign + their attack windows)", fontsize=8, va="top")
ax.set_xlabel("windows (log scale)")
ax.set_title("all/T10_benign: 80 / 10 / 10 by sender vehicle over all 8 folders (fixed on disk)")
plt.show()

Load the two splits used for fitting (from the decoded-shard cache when present) and check them.

In [ ]:
splits = {
    name: WindowDataset.load(settings.data_dir, name, settings.workers, cache_dir=settings.cache_dir)
    for name in ("train", "val")
}
train, val = splits["train"], splits["val"]
assert not (set(train.vehicles) & set(val.vehicles)), "a sender vehicle in train and val"
for d in (train, val):
    assert (d.info["label"].astype(int) == 0).all() and (d.info["is_attack"].astype(int) == 0).all()
    assert len(d) == META["counts_by_split"][d.split]["windows"]
print("train and val: benign only, no sender vehicle in both, window counts equal metadata")
pd.DataFrame(
    {
        d.split: {
            "windows": len(d),
            "tracks": len(set(d.tracks)),
            "sender vehicles": len(set(d.vehicles)),
            "scenario folders": len(set(d.info["scenario"])),
        }
        for d in (train, val)
    }
).T

Shape: each window is up to 10 messages (1 Hz) of one link × 13 normalised features, padded to 10 rows; padding
rows are zero and a prefix mask marks the real rows. The model gets the 13 features plus the mask as a 14th channel:
**14 × 10** per window. Links longer than 10 messages are cut into consecutive non-overlapping crops (the last one
may be short).

In [ ]:
print(
    "window x:",
    train.x.shape[1:],
    "+ mask:",
    train.mask.shape[1:],
    "-> model input:",
    (settings.n_features + 1, settings.seq_len),
)
rows = []
for name, d in splits.items():
    n = d.n_messages
    rows.append(
        {
            "split": name,
            "min": n.min(),
            "median": np.median(n),
            "mean": round(n.mean(), 2),
            "max": n.max(),
            f"share = {settings.seq_len}": round(float((n == settings.seq_len).mean()), 3),
            "share 1-3": round(float((n <= 3).mean()), 3),
            "padding share": round(1 - d.mask.mean(), 3),
        }
    )
display(pd.DataFrame(rows).set_index("split"))
print(
    f"padding over all 7 splits: {META['padding']['overall']:.3f}; "
    f"1-3-row share over all 7 splits: {META['padding']['share_1_3_overall']:.3f}"
)

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.hist(train.n_messages, bins=np.arange(1, settings.seq_len + 2) - 0.5, color="tab:blue")
ax.set_xlabel("real messages per window")
ax.set_ylabel("train windows")
ax.set_yscale("log")
ax.set_title(f"Benign train: real messages per window (peak at {settings.seq_len} from the cut long links)")
plt.show()

Per-feature statistics over the **real** train rows (normalised with the benign train mean / std, so ≈ 0 / 1).

In [ ]:
real = train.x[train.mask]  # (real rows, 13)
pd.DataFrame({"mean": real.mean(0), "std": real.std(0), "min": real.min(0), "max": real.max(0)}, index=FEATURES).round(
    3
)

One train window: the first full (10-row) window of the train split, as a heat map of its 13 features.

In [ ]:
i = int(np.flatnonzero(train.n_messages == settings.seq_len)[0])
fig, ax = plt.subplots(figsize=(10, 4))
im = ax.imshow(train.x[i].T, aspect="auto", cmap="coolwarm", vmin=-3, vmax=3)
ax.set_yticks(range(len(FEATURES)), FEATURES)
ax.set_xlabel("message (time step, 1 Hz)")
ax.set_title(f"window {train.ids[i]} ({train.info['scenario'][i]}): {train.n_messages[i]} real rows", fontsize=9)
fig.colorbar(im, ax=ax, label="normalised value")
plt.show()

## 2. Model

Layers with their output shapes for one batch of 2 windows, and the trainable parameter count (asserted against the
package constant). 1 stride-2 stage(s): encoder strides 1, 2, 1; decoder ConvTranspose k3 s1 (length kept), k4 s2; latent length 5.

In [ ]:
model = ConvAutoencoder(settings)
x0, m0 = torch.zeros(2, settings.seq_len, settings.n_features), torch.ones(2, settings.seq_len, dtype=torch.bool)
table = []
layer_types = (torch.nn.Conv1d, torch.nn.ConvTranspose1d, torch.nn.Linear, torch.nn.BatchNorm1d)
hooks = [
    mod.register_forward_hook(
        lambda mod, inp, out, name=name: table.append(
            (name, type(mod).__name__, tuple(out.shape[1:]), sum(p.numel() for p in mod.parameters()))
        )
    )
    for name, mod in model.named_modules()
    if isinstance(mod, layer_types)
]
model.eval()
with torch.no_grad():
    recon0, z0 = model(x0, m0)
for h in hooks:
    h.remove()
display(pd.DataFrame(table, columns=["layer", "type", "output (per window)", "params"]))
params = count_params(model)
assert params == TOTAL_PARAMS, (params, TOTAL_PARAMS)
print("reconstruction:", tuple(recon0.shape[1:]), " z:", tuple(z0.shape[1:]))
print(f"trainable parameters: {params:,} (= TOTAL_PARAMS)")

## 3. Training record

What `task.autoencoder_benign_t10.train` wrote: the summary (thresholds shown in section 5), the per-epoch history
(train vs validation masked MSE, best epoch, LR drops; then the gradient L2 norm, mean and max over the epoch's
batches, before clipping, and the LR) and log lines.

In [ ]:
summary = json.loads((RUN / "summary.json").read_text())
print(json.dumps({k: v for k, v in summary.items() if k != "thresholds"}, indent=2))
history = pd.read_json(RUN / "history.jsonl", lines=True)
fig, (ax, ax2) = plt.subplots(2, 1, figsize=(9, 7), sharex=True, height_ratios=(3, 2))
ax.plot(history.epoch, history.train_loss, label="train (benign)")
ax.plot(history.epoch, history.val_loss, label="validation (benign)")
for a in (ax, ax2):
    for k, c in enumerate(summary["lr_changes"]):
        a.axvline(c["after_epoch"] + 0.5, color="grey", ls=":", label="LR halved" if k == 0 else None)
    a.axvline(summary["best_epoch"], color="red", ls="--", label=f"best epoch {summary['best_epoch']}")
ax.set_yscale("log")
ax.set_ylabel("masked MSE (real rows)")
ax.legend()
ax.set_title(f"{RUN.name}: loss per epoch ({summary['stop_reason']})")
ax2.plot(history.epoch, history.grad_norm_mean, color="tab:green", label="gradient L2 norm, mean")
ax2.plot(history.epoch, history.grad_norm_max, color="tab:olive", ls=":", label="gradient L2 norm, max")
ax2.set_ylabel("gradient L2 norm (pre-clip)")
ax2.set_xlabel("epoch")
lr_ax = ax2.twinx()
lr_ax.plot(history.epoch, history.lr, color="tab:purple", label="LR")
lr_ax.set_yscale("log")
lr_ax.set_ylabel("learning rate")
h1, l1 = ax2.get_legend_handles_labels()
h2, l2 = lr_ax.get_legend_handles_labels()
ax2.legend(h1 + h2, l1 + l2, fontsize=8)
fig.tight_layout()
plt.show()
display(history.iloc[[0, 1, summary["best_epoch"] - 1, len(history) - 1]].set_index("epoch").round(5))
log = (RUN / "train.log").read_text().splitlines()
print("\n".join(log[:6] + ["..."] + [l for l in log if "*best" in l][-2:] + ["..."] + log[-4:-1]))

## 4. Reconstruction on val

Load `best.pt` and rebuild the benign val windows: three full val windows, input vs reconstruction for three features;
then the mean reconstruction error per feature over val.

In [ ]:
trainer = Trainer.from_checkpoint(RUN / "best.pt")
cfg = trainer.settings
val_out = trainer.score(val.x, val.mask, keep_recon=True)
pooled = float((val_out["error"] * val.n_messages).sum() / val.n_messages.sum())
print(
    f"val windows {len(val):,}; mean window error {val_out['error'].mean():.4f}; "
    f"pooled over real rows {pooled:.4f} (run's best val loss {summary['best_val_loss']:.4f})"
)

show = ["claimed_pos_x", "claimed_vel_x", "range"]
picks = np.flatnonzero(val.n_messages == settings.seq_len)[:3]
fig, axes = plt.subplots(len(picks), len(show), figsize=(12, 2.6 * len(picks)), squeeze=False)
for r, i in enumerate(picks):
    n = val.n_messages[i]
    for c, f in enumerate(show):
        j = FEATURES.index(f)
        axes[r, c].plot(val.x[i, :n, j], label="input")
        axes[r, c].plot(val_out["recon"][i, :n, j], "--", label="reconstruction")
        axes[r, c].set_title(f"{f} ({n} rows)", fontsize=9)
axes[0, 0].legend(fontsize=8)
fig.tight_layout()
plt.show()

pd.Series(val_out["error_feature"].mean(0), index=FEATURES, name="val MSE").sort_values(ascending=False).round(
    4
).to_frame()

## 5. Test

**Scoring first, labels after.** The Mahalanobis model stored by the run (`mahalanobis.npz`, fitted on the benign
train z) is checked against a refit here. Thresholds = 95th / 99th percentile of the **benign val** scores (window
level) or of the val track scores (track level); they are recomputed and compared with the run's summary. Then every
window of the benign test split and of the four attack test splits is scored (reconstruction error, Mahalanobis
distance, per-track max / mean) before any label or split name enters a metric.

In [ ]:
maha = Mahalanobis.from_state(np.load(RUN / "mahalanobis.npz"))
train_z = trainer.score(train.x, train.mask)["z"]
refit = Mahalanobis(cfg.mahalanobis_max_cond, cfg.mahalanobis_shrinkage).fit(train_z)
print(
    f"mahalanobis: cond {maha.cond:.3g}, shrinkage {maha.shrinkage:g}; refit on train z matches: "
    f"{np.allclose(maha.mean, refit.mean, atol=1e-5) and np.allclose(maha.precision, refit.precision, rtol=1e-3, atol=1e-4)}"
)

LEVELS = ["window", "track max", "track mean"]
SCORES = ["reconstruction", "mahalanobis"]


def score_split(d, out=None):
    # Window scores and per-track max / mean of one split; no labels involved.
    out = out if out is not None else trainer.score(d.x, d.mask)
    tr = TrackScores(d.tracks)
    res = {"tracks": tr, "error_feature": out["error_feature"], "z": out["z"], "n_messages": d.n_messages}
    for name, s in (("reconstruction", out["error"]), ("mahalanobis", maha.score(out["z"]))):
        res[(name, "window")] = s
        agg = tr.aggregate(s)
        res[(name, "track max")], res[(name, "track mean")] = agg["max"], agg["mean"]
    return res


scored = {"val": score_split(val, val_out)}
thresholds = {
    key: quantile_thresholds(scored["val"][key], cfg.threshold_quantiles)
    for key in [(n, l) for n in SCORES for l in LEVELS]
}
run_thr = summary["thresholds"]
same = all(
    np.isclose(thresholds[(n, "window")][q], run_thr["window"][n][q])
    and np.isclose(thresholds[(n, "track max")][q], run_thr["track"][f"{n}_max"][q])
    and np.isclose(thresholds[(n, "track mean")][q], run_thr["track"][f"{n}_mean"][q])
    for n in SCORES
    for q in ("p95", "p99")
)
print("val thresholds equal the run's summary:", same)
display(pd.DataFrame({f"{n} / {l}": thresholds[(n, l)] for n in SCORES for l in LEVELS}).T.round(4))

In [ ]:
test_data = {}
for name in EVAL_SPLITS:
    d = WindowDataset.load(settings.data_dir, name, settings.workers, cache_dir=settings.cache_dir)
    scored[name] = score_split(d)
    test_data[name] = d
    print(f"scored {name}: {len(d):,} windows, {len(scored[name]['tracks']):,} tracks")

Labels are read now. Sanity checks: the benign test split holds label 0 only and each attack split only its own attack
code; test vehicles are disjoint from train / val; no track appears in two splits (so every track has one label).
Positive = attack. Each attack split is evaluated against the **same benign test windows**.

In [ ]:
CODES = {"test_gridsybil": 16, "test_datareplay": 17, "test_dosrandom": 18, "test_dosdisruptive": 19}
assert (test_data["test"].info["label"].astype(int) == 0).all()
for name, code_ in CODES.items():
    assert (test_data[name].info["label"].astype(int) == code_).all(), name
test_veh = set().union(*(set(d.vehicles) for d in test_data.values()))
assert not (test_veh & set(train.vehicles)) and not (test_veh & set(val.vehicles))
track_sets = {n: set(scored[n]["tracks"].tracks) for n in EVAL_SPLITS}
overlap = sum(len(track_sets[a] & track_sets[b]) for i, a in enumerate(EVAL_SPLITS) for b in EVAL_SPLITS[i + 1 :])
print("labels as expected; test vehicles disjoint from train / val; tracks shared between test splits:", overlap)
attack_veh = set().union(*(set(test_data[n].vehicles) for n in CODES))
print(
    f"test vehicles: benign {len(set(test_data['test'].vehicles)):,}, attack {len(attack_veh):,}, "
    f"attack vehicles that also send benign test windows {len(attack_veh & set(test_data['test'].vehicles)):,}"
)

Per attack vs benign test: AUROC, PR-AUC (chance = attack share of the pair), and at the val-p95 / val-p99 thresholds
the attack recall and the benign-test FPR, per window and per track, for both scores. A **length-only** reference
(score = − real messages per window; direction fixed in advance by F11, not chosen on test) shows how much of each pair
window length alone separates. Per window, the AUROC is also given on the windows with **at least 4 real rows** only
(both classes restricted), where length no longer dominates.

In [ ]:
def pair(attack, key):
    b, a = scored["test"][key], scored[attack][key]
    return np.r_[np.zeros(len(b), int), np.ones(len(a), int)], np.r_[b, a]


long4 = {a: np.r_[scored["test"]["n_messages"], scored[a]["n_messages"]] >= 4 for a in ATTACKS}
rows = []
for attack, short in ATTACKS.items():
    for name in SCORES:
        for level in LEVELS:
            y, s = pair(attack, (name, level))
            r = {
                "attack": short,
                "score": name,
                "level": level,
                "AUROC": roc_auc_score(y, s),
                "PR-AUC": average_precision_score(y, s),
                "chance": y.mean(),
                "n benign": int((y == 0).sum()),
                "n attack": int(y.sum()),
            }
            if level == "window":
                r["AUROC (>= 4 rows)"] = roc_auc_score(y[long4[attack]], s[long4[attack]])
                r["n attack (>= 4 rows)"] = int(y[long4[attack]].sum())
            for q in ("p95", "p99"):
                th = thresholds[(name, level)][q]
                r[f"recall {q}"] = float((s[y == 1] > th).mean())
                r[f"benign FPR {q}"] = float((s[y == 0] > th).mean())
            rows.append(r)
    y = np.r_[np.zeros(len(scored["test"]["n_messages"]), int), np.ones(len(scored[attack]["n_messages"]), int)]
    s = -np.r_[scored["test"]["n_messages"], scored[attack]["n_messages"]].astype(float)
    rows.append(
        {
            "attack": short,
            "score": "length only (reference)",
            "level": "window",
            "AUROC": roc_auc_score(y, s),
            "PR-AUC": average_precision_score(y, s),
            "chance": y.mean(),
            "n benign": int((y == 0).sum()),
            "n attack": int(y.sum()),
        }
    )
res = pd.DataFrame(rows)
res.set_index(["attack", "score", "level"]).round(3)

AUROC at a glance (rows = score / level, columns = attack).

In [ ]:
auroc = res.pivot_table(index=["score", "level"], columns="attack", values="AUROC", sort=False)[list(ATTACKS.values())]
auroc.round(3)

Score histograms per window, benign test vs each attack, with the val-p95 threshold.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for r, name in enumerate(SCORES):
    allv = np.concatenate([scored[n][(name, "window")] for n in EVAL_SPLITS])
    bins = np.logspace(np.log10(max(allv.min(), 1e-5)), np.log10(allv.max()), 60)
    for c, (attack, short) in enumerate(ATTACKS.items()):
        ax = axes[r, c]
        ax.hist(scored["test"][(name, "window")], bins=bins, alpha=0.6, density=True, label="benign test")
        ax.hist(scored[attack][(name, "window")], bins=bins, alpha=0.6, density=True, label=short, color="tab:red")
        ax.axvline(thresholds[(name, "window")]["p95"], color="black", ls="--", lw=1, label="val p95")
        y, s = pair(attack, (name, "window"))
        ax.set_xscale("log")
        ax.set_title(f"{name}: {short}, AUROC {roc_auc_score(y, s):.3f}", fontsize=10)
        ax.set_xlabel("masked MSE" if name == "reconstruction" else "Mahalanobis distance of z")
        ax.legend(fontsize=7)
fig.tight_layout()
plt.show()

Reconstruction error vs real messages per window (median per length), benign test and each attack: short windows are
where most DataReplay / DoS windows sit (F11).

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
for n, lab in [("test", "benign test"), *ATTACKS.items()]:
    df = pd.DataFrame({"n": scored[n]["n_messages"], "e": scored[n][("reconstruction", "window")]})
    g = df.groupby("n").e.median()
    ax.plot(g.index, g.values, marker=".", label=lab)
ax.set_yscale("log")
ax.set_xlabel("real messages per window")
ax.set_ylabel("median reconstruction error")
ax.legend(fontsize=8)
ax.set_title("Reconstruction error by window length")
plt.show()

Mean per-feature reconstruction error by class on the test windows, and the per-feature AUROC of each attack vs benign test.

In [ ]:
ef = {n: scored[n]["error_feature"] for n in EVAL_SPLITS}
tab = pd.DataFrame({("mean error", "benign"): ef["test"].mean(0)}, index=FEATURES)
for attack, short in ATTACKS.items():
    tab[("mean error", short)] = ef[attack].mean(0)
for attack, short in ATTACKS.items():
    y = np.r_[np.zeros(len(ef["test"]), int), np.ones(len(ef[attack]), int)]
    e = np.r_[ef["test"], ef[attack]]
    tab[("AUROC", short)] = [roc_auc_score(y, e[:, j]) for j in range(e.shape[1])]
tab.columns = pd.MultiIndex.from_tuples(tab.columns)
tab.round(3)

The windows with at least 4 real rows per split: how many, and the median normalised log-Δτ of their real rows (benign train mean / std scale).

In [ ]:
rows = {}
for n, lab in [("test", "benign test"), *ATTACKS.items()]:
    d = test_data[n]
    sel = d.n_messages >= 4
    rows[lab] = {
        "windows >= 4 rows": int(sel.sum()),
        "share": round(float(sel.mean()), 4),
        "median log_dtau (real rows)": float(np.median(d.x[sel][d.mask[sel]][:, FEATURES.index("log_dtau")])),
    }
pd.DataFrame(rows).T.round(3)

2-D PCA of the latent z (fitted on the benign test z; 4,000 windows per class; inspection only).

In [ ]:
rng = np.random.default_rng(0)
pca = PCA(2, random_state=0).fit(scored["test"]["z"])
fig, ax = plt.subplots(figsize=(7, 6))
for n, lab in [("test", "benign test"), *ATTACKS.items()]:
    z = scored[n]["z"]
    zz = pca.transform(z[rng.choice(len(z), size=min(4000, len(z)), replace=False)])
    ax.scatter(zz[:, 0], zz[:, 1], s=3, alpha=0.35, label=lab)
ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.0%} of benign z variance)")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.0%})")
ax.legend(markerscale=4, fontsize=8)
ax.set_title("Test z, first two principal components of the benign test z")
plt.show()

## 6. Context (not a fair comparison)

Other autoencoders in `task/`, GridSybil (and, for T = 32, every attack) vs benign window AUROC on their own test
splits:

- the same one-class protocol at **T = 32** (`task/autoencoder_benign_t32`, same split, other window length),
  reconstruction, mean of seeds 0–3 (agent.md, 2026-11-17): GridSybil **0.7335**, DataReplay 0.387, DoSRandom
  0.967, DoSDisruptive 0.598;
- unsupervised ConvAutoencoders trained on benign + GridSybil as "normal" (`task/autoencoder`,
  `task/autoencoder_overlap`, T = 48): reconstruction AUROC **0.484**;
- supervised autoencoder with a classifier head (`task/autoencoder_st`, T = 48, GridSybil labels in training):
  AUROC **0.929** (GridSybil only, says nothing about unseen attacks).

Different window length, data, split or training signal: these numbers frame the result, they do not rank the models.

In [ ]:
ctx = res[(res.level == "window") & (res.score != "length only (reference)")].pivot_table(
    index="score", columns="attack", values="AUROC", sort=False
)[list(ATTACKS.values())]
ctx.loc["task/autoencoder_benign_t32, T32, one-class, reconstruction, seeds 0-3 (context)"] = [
    0.7335,
    0.387,
    0.967,
    0.598,
]
ctx.loc["task/autoencoder(_overlap), T48, unsupervised (context)", "GridSybil"] = 0.484
ctx.loc["task/autoencoder_st, T48, supervised (context)", "GridSybil"] = 0.929
ctx.round(3)

The numbers the analysis below refers to.

In [ ]:
results = {
    "run": RUN.name,
    "params": summary["params"],
    "best_epoch": summary["best_epoch"],
    "best_val_loss": round(summary["best_val_loss"], 5),
    "epochs_run": summary["epochs_run"],
    "stop_reason": summary["stop_reason"],
    "lr_changes": len(summary["lr_changes"]),
}
for _, r in res.iterrows():
    keys = [
        k
        for k in (
            "AUROC",
            "AUROC (>= 4 rows)",
            "PR-AUC",
            "chance",
            "recall p95",
            "benign FPR p95",
            "recall p99",
            "benign FPR p99",
        )
        if k in r and pd.notna(r[k])
    ]
    results[f"{r.attack}/{r.score}/{r.level}"] = {k: round(float(r[k]), 3) for k in keys}
results

## 7. Analysis

**Not written yet.** This notebook was copied from the T = 32 notebook before any T = 10 run existed; the T = 32
analysis text was removed because its numbers do not describe this run (RULE 1). Write the analysis from the
outputs above after `npm run aeb10:train` has finished (one seed per run folder, no confidence interval; caveats F1, F11, F13,
F14 as in the T = 32 notebook).